# User Notebook

## Introduction

The library supports **data synthesis** based on target maps. This notebook demonstrates how to define **custom loss functions** that compare statistical descriptors, and how to **run optimization procedures** to generate signals or fields that match desired statistical properties.

Examples are provided to illustrate typical workflows and recommended usage patterns.

Note:

- For **2D Kernel** data: **mean**, **variance**, **scattering coefficients**, and **power spectrum** are all optimized (periodic or non-periodic data; with or without NaN values).

- For **2D FFT** data: **mean**, **variance**, **scattering coefficients**, and **power spectrum** are all optimized (periodic or non-periodic data; NaN values are non handled).

**Power spectrum** is not optimized yet if there are NaN values in either the target or the running map.It should be possible in the future for the kernel dataclass with an implementation of power spectrum calculation via convolution.

During synthesis to generate a non-PBC image, there will be pixels located at the edges of the image that are not optimized because they do not contribute to the calculation of scattering statistics. To obtain a non-PBC image that preserves the correct dimensions of the target, run the synthesis on an image twice as large and then crop the result.

During the synthesis, it is specified into user-level wrapper an ST operator for the target and an ST operator for the running map. This is particularly useful for syntheses with NaNs, as it allows specifying two different masks: one for the target and one for the running map. For other syntheses, the ST operator will be the same for both.

For synthesis from maps, it is currently not possible to perform synthesis on maps with sizes different from that of the target.

In [ ]:
# Import useful libraries, test data path and STL modules

# Library imports
import numpy as np
import matplotlib.pyplot as plt
import torch

import os
from pathlib import Path
import sys

from types import SimpleNamespace


# Add parent directory to sys.path to import STL package modules
os.chdir("/obs/dtibi/STL-Dev/docs/user_notebook")
PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(PARENT_DIR)
print("Parent directory added to sys.path:", ".../" + os.path.basename(PARENT_DIR))

# Path to test data
DATA_TEST_PATH = PARENT_DIR + "/data" + "/test"
print(
    "Dataset directory used:",
    ".../" + os.path.basename(PARENT_DIR) + DATA_TEST_PATH.split(os.path.basename(PARENT_DIR))[-1]
)

# STL package imports
from STL_main.STL_2D_Kernel_Torch import STL_2D_Kernel_Torch, WaveletOperator2Dkernel_torch
from STL_main.STL_2D_FFT_Torch import STL_2D_FFT_Torch, WaveletOperator2D_FFT_torch
from STL_main.Synthesis import synthesize_from_maps, synthesize_from_stats
from STL_main.torch_backend import _DEFAULT_DEVICE

print("Working on device:", _DEFAULT_DEVICE)

In [ ]:
# command to auto-reload modules when they are edited (easier for testing and debugging)
%load_ext autoreload
%autoreload 2

In [ ]:
def plot_target_vs_synthesis(target, synthesis, suptitle, titles=["Target", "Synthesis"]):
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

    t = target.cpu().numpy()
    u = synthesis.cpu().numpy()

    # Color scale limits of the target image, ignoring NaNs
    mean_t = np.nanmean(t)
    std_t = np.nanstd(t)

    vmin = mean_t - 3 * std_t
    vmax = mean_t + 3 * std_t

    im0 = axes[0].imshow(t, cmap="plasma", vmin=vmin, vmax=vmax)
    axes[0].axis("off")
    axes[0].set_title(titles[0])

    im1 = axes[1].imshow(u, cmap="plasma", vmin=vmin, vmax=vmax)
    axes[1].axis("off")
    axes[1].set_title(titles[1])

    fig.colorbar(im0, ax=axes, shrink=0.7)

    fig.suptitle(suptitle, fontsize=16)

    plt.show()

In [ ]:
# Image loading
im = np.load(DATA_TEST_PATH + "/" + 'Turb_6.npy')[:, None, :, :] # [Nb, Nc, Nx, Ny]

# Display original image
im_plot = plt.imshow(
    im[0, 0, :, :],
    cmap="plasma",
    vmin=np.nanmean(im) - 3 * np.nanstd(im),
    vmax=np.nanmean(im) + 3 * np.nanstd(im)
)
plt.title("Original Image")
plt.axis('off')
plt.colorbar(im_plot)
plt.tight_layout()
plt.show()

## Synthesis

**The synthesis table below** summarizes all the syntheses performed in this certification notebook. It is not exhaustive: for example, it would be possible to perform FFT syntheses for many categories (except for syntheses where at least the target or the running map contains a mask).

| Category | Input image(s) | Output image(s) | Channel(s) | Method(s) |
|----------|---------------|-----------------|---------|--------|
| One → One | 1 PBC image | 1 PBC image | Mono + Cross | Kernel + FFT |
| One → One (Different Shape) | 1 PBC image | 1 PBC image | Mono | Kernel |
| One → Many | 1 PBC image | N PBC images | Mono | Kernel |
| Many → Many (N ≠ M) | N PBC image | M PBC images | Mono | Kernel |
| Many → Many (N = M) | N PBC image | N PBC images | Mono | Kernel |
| PBC → Non PBC | 1 PBC image | 1 non-PBC image | Mono | Kernel |
| Non PBC → PBC | 1 non-PBC image | 1 PBC image | Mono | Kernel |
| Non PBC → Non PBC | 1 non-PBC image | 1 non-PBC image | Mono | Kernel |
| NaNs → NaNs | 1 image with NaNs | 1 image with NaNs | Mono | Kernel |
| NaNs → NaNs (≠ Mask)| 1 image with NaNs  | 1 image with NaNs | Mono | Kernel |
| Non NaNs → NaNs | 1 image without NaNs | 1 image with NaNs | Mono | Kernel |
| NaNs → Non NaNs | 1 image with NaNs | 1 image without NaNs | Mono | Kernel |


#### 1. One → One 

##### 1.1 Kernel - Mono Channel

In [ ]:
im_target = im[0, 0, :, :] # Mono channel image

# Instantiate kernel data class on the target image
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
u_kernel = synthesize_from_maps(
    data_target_kernel,
    pbc_running=True,
    nbatch=1,
    max_iter=100
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel.array, u_kernel, suptitle = "Kernel", titles=["Target", "Synthesis"])

Example code for performing synthesis from statistics

In [ ]:
im_target = im[0, 0, :, :] # Mono channel image

# Instantiate kernel data class and st opertor
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)
st_op_kernel = data_target_kernel.get_ST_op()

# Standardize target data
data_target_kernel_std, mean_target, std_target = st_op_kernel.wavelet_op.standardize(data_target_kernel, mean_field=False, inplace=False)

# Compute target statistics
target_stats_kernel = st_op_kernel.apply(data_target_kernel_std)

# Call user-friendly synthesis wrapper
u_kernel = synthesize_from_stats(
    target_stats_kernel,
    pbc_running=True,
    mean_target=mean_target,
    std_target=std_target,
    max_iter=100
)

# Plot target vs synthesis
plot_target_vs_synthesis(data_target_kernel.array, u_kernel, suptitle = "Kernel", titles=["Target", "Synthesis"])

##### 1.2 FFT - Mono Channel

Since the power spectrum is systematically included in the synthesis optimization, it is necessary in mono-channel FFT to increase the number of iterations, as the power spectrum dominates the scattering coefficients in the loss.

In [ ]:
im_target = im[0, 0, :, :] # Mono channel image

# Instantiate FFT data class on the target image
data_target_fft = STL_2D_FFT_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
u_fft = synthesize_from_maps(
    data_target_fft,
    nbatch=1,
    pbc_running=True,
    max_iter=100
)

In [ ]:
plot_target_vs_synthesis(data_target_fft.array, u_fft, suptitle = "FFT", titles=["Target", "Synthesis"])

Example code for performing synthesis from statistics

In [ ]:
im_target = im[0, 0, :, :] # Mono channel image

# Instantiate fft data class and st opertor
data_target_fft = STL_2D_FFT_Torch(im_target, pbc=True)
st_op_fft = data_target_fft.get_ST_op()

# Standardize target data
data_target_fft_std, mean_target, std_target = st_op_fft.wavelet_op.standardize(data_target_fft, mean_field=False, inplace=False)

# Compute target statistics
target_stats_fft = st_op_fft.apply(data_target_fft_std)

# Call user-friendly synthesis wrapper
u_fft = synthesize_from_stats(
    target_stats_fft,
    pbc_running=True,
    mean_target=mean_target,
    std_target=std_target,
    max_iter=100
)

# Plot target vs synthesis
plot_target_vs_synthesis(data_target_fft.array, u_fft, suptitle = "FFT", titles=["Target", "Synthesis"])

##### 1.3 Kernel - Multi Channels

In [ ]:
im_target = np.stack([im[0, 0, :, :], (im[0 ,0, :, :]-im[0, 0, :, :].mean(keepdims=True))**2]) # Constructing an image with 2 channels
im_target -= im_target.mean(axis=(1,2), keepdims=True)

# Instantiate kernel data class on the multi-channel target image
data_target_kernel_multi_channels = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
compute_cross_matrix = torch.tensor([[1, 1], [0, 1]], dtype=torch.bool)
u_kernel_multi_channels = synthesize_from_maps(
    data_target_kernel_multi_channels,
    pbc_running=True,
    nbatch=1,
    compute_cross_matrix=compute_cross_matrix
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel_multi_channels.array[0], u_kernel_multi_channels[0], suptitle = "Kernel - Multi Channels", titles=["Target - Channel 1", "Synthesis - Channel 1"])
plot_target_vs_synthesis(data_target_kernel_multi_channels.array[1], u_kernel_multi_channels[1], suptitle = "Kernel - Multi Channels", titles=["Target - Channel 2", "Synthesis - Channel 2"])

Example code for performing synthesis from statistics

In [ ]:
im_target = np.stack([im[0, 0, :, :], (im[0 ,0, :, :]-im[0, 0, :, :].mean(keepdims=True))**2]) # Constructing an image with 2 channels
im_target -= im_target.mean(axis=(1,2), keepdims=True)

# Instantiate kernel data class and st opertor on the multi-channel target image
data_target_kernel_multi_channels = STL_2D_FFT_Torch(im_target, pbc=True)
st_op_kernel = data_target_kernel_multi_channels.get_ST_op()

# Standardize target data
data_target_kernel_std, mean_target, std_target = st_op_kernel.wavelet_op.standardize(data_target_kernel_multi_channels, mean_field=False, inplace=False)

# Compute target statistics
# NOTE: Parameters such as `compute_cross_matrix` or `has_fewer_convolution` should NOT be passed to the `synthesis_from_stats` wrapper.
# They must be specified ONLY here, during the computation of the target statistics via `st_op_kernel.apply`
compute_cross_matrix = torch.tensor([[1, 1], [0, 1]], dtype=torch.bool)
target_stats_kernel = st_op_kernel.apply(data_target_kernel_std, compute_cross_matrix=compute_cross_matrix)

# Call user-friendly synthesis wrapper
u_kernel_multi_channels = synthesize_from_stats(
    target_stats_kernel,
    pbc_running=True,
    mean_target=mean_target,
    std_target=std_target,
    max_iter=100
)

# Plot target vs synthesis for each channel
plot_target_vs_synthesis(data_target_kernel_multi_channels.array[0], u_kernel_multi_channels[0], suptitle = "Kernel - Multi Channels", titles=["Target - Channel 1", "Synthesis - Channel 1"])
plot_target_vs_synthesis(data_target_kernel_multi_channels.array[1], u_kernel_multi_channels[1], suptitle = "Kernel - Multi Channels", titles=["Target - Channel 2", "Synthesis - Channel 2"])

##### 1.4 FFT - Multi Channels

In [ ]:
im_target = np.stack([im[0, 0, :, :], (im[0 ,0, :, :]-im[0, 0, :, :].mean(keepdims=True))**2]) # Constructing an image with 2 channels
im_target -= im_target.mean(axis=(1,2), keepdims=True)

# Instantiate FFT data class on the multi-channel target image
data_target_fft_multi_channels = STL_2D_FFT_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
compute_cross_matrix = torch.tensor([[1, 1], [0, 1]], dtype=torch.bool)
u_fft_multi_channels = synthesize_from_maps(
    data_target_fft_multi_channels,
    pbc_running=True,
    nbatch=1,
    compute_cross_matrix=compute_cross_matrix,
    max_iter=100
)

In [ ]:
plot_target_vs_synthesis(data_target_fft_multi_channels.array[0], u_fft_multi_channels[0], suptitle = "FFT - Multi Channels", titles=["Target - Channel 1", "Synthesis - Channel 1"])
plot_target_vs_synthesis(data_target_fft_multi_channels.array[1], u_fft_multi_channels[1], suptitle = "FFT - Multi Channels", titles=["Target - Channel 2", "Synthesis - Channel 2"])

#### One → One (different shape)

In [ ]:
im_target = im[0, 0, :, :] # Target image

# Instantiate kernel data class on the target image
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
u_kernel_128 = synthesize_from_maps(
    data_target_kernel,
    pbc_running=True,
    nbatch=1,
    running_shape=(128, 128)
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel.array, u_kernel_128, suptitle = "Kernel - Smaller Synthesis", titles=["Target - (256, 256)", "Synthesis - (128, 128)"])

In [ ]:
im_target = im[0, 0, :, :] # Target image

# Instantiate kernel data class on the target image
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
u_kernel_512 = synthesize_from_maps(
    data_target_kernel,
    pbc_running=True,
    nbatch=1,
    running_shape=(512, 512)
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel.array, u_kernel_512, suptitle = "Kernel - Bigger Synthesis", titles=["Target - (256, 256)", "Synthesis - (512, 512)"])

#### One → Many

In [ ]:
im_target = im[0, 0, :, :]

# Instantiate kernel data class on the target image
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
nbatch = 4
u_kernel_many_batchs = synthesize_from_maps(
    data_target_kernel,
    pbc_running=True,
    nbatch=nbatch
)

In [ ]:
to_numpy = lambda x: x.detach().cpu().numpy()

t = to_numpy(data_target_kernel.array)
u_list = [to_numpy(u) for u in u_kernel_many_batchs[:4]]

mean_t = np.nanmean(t)
std_t = np.nanstd(t)

vmin = mean_t - 3 * std_t
vmax = mean_t + 3 * std_t

# --- Target ---
fig_target, ax = plt.subplots(figsize=(8, 8))
im0 = ax.imshow(t, cmap="plasma", vmin=vmin, vmax=vmax)
ax.axis("off")
ax.set_title("Target")
fig_target.colorbar(im0, ax=ax, shrink=0.7)

# --- Synthesis ---
fig_synthesis, axes = plt.subplots(2, 2, figsize=(8, 6))

for i, (ax, u) in enumerate(zip(axes.flat, u_list), start=1):
    im0 = ax.imshow(u, cmap="plasma", vmin=vmin, vmax=vmax)
    ax.axis("off")
    ax.set_title(f"Synthesis {i}")

fig_synthesis.colorbar(im0, ax=axes.ravel().tolist(), shrink=0.7)

plt.show()

#### Many -> Many (N ≠ M)

In this scenario, the `mean_field` argument of the `synthesize_from_maps` synthesis wrapper is left at its default value. This means that when optimizing the statistics, they are averaged over the **batch dimension**, rather than computed **map by map**.

In [ ]:
im_target = np.expand_dims(im[:4, 0, :, :], axis=1)

# Instantiate kernel data class on the target image
data_target_kernel_many_batchs = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
nbatch = 6
u_kernel_many_batchs = synthesize_from_maps(
    data_target_kernel_many_batchs,
    pbc_running=True,
    nbatch=nbatch,
)

In [ ]:
to_numpy = lambda x: x.detach().cpu().numpy()

t_list = [to_numpy(t) for t in data_target_kernel_many_batchs.array[:4, 0]]  # 4 maps  
u_list = [to_numpy(u) for u in u_kernel_many_batchs[:6, 0]]  # 6 maps

# Flatten target arrays to compute global vmin/vmax
mean_t = np.nanmean(t_list)
std_t = np.nanstd(t_list)

vmin = mean_t - 3 * std_t
vmax = mean_t + 3 * std_t

# --- Plot Target Maps ---
fig_target, axes = plt.subplots(2, 2, figsize=(8, 6))

for ax, t, i in zip(axes.flatten(), t_list, range(4)):
    im0 = ax.imshow(t, cmap="plasma", vmin=vmin, vmax=vmax)
    ax.axis("off")
    ax.set_title(f"Target {i+1}")

fig_target.colorbar(im0, ax=axes, shrink=0.7)
plt.suptitle("Target Maps")

# --- Plot Synthesis Maps ---
fig_synth, axes = plt.subplots(2, 3, figsize=(10, 6))

for ax, u, i in zip(axes.flatten(), u_list, range(6)):
    im0 = ax.imshow(u, cmap="plasma", vmin=vmin, vmax=vmax)
    ax.axis("off")
    ax.set_title(f"Synthesis {i+1}")

fig_synth.colorbar(im0, ax=axes, shrink=0.7)
plt.suptitle("Synthesis Maps")

plt.show()

#### Many -> Many

In this scenario, the `mean_field` argument of the `synthesize_from_maps` synthesis wrapper is set to `False`. This means that the statistics are optimized **map by map**, so it is necessary to have the **same batch size** for both the target and the running data.

In [ ]:
im_target = np.expand_dims(im[:4, 0, :, :], axis=1)

# Instantiate kernel data class on the target image
data_target_kernel_many_batchs = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
nbatch = 4
u_kernel_many_batchs = synthesize_from_maps(
    data_target_kernel_many_batchs,
    pbc_running=True,
    nbatch=nbatch,
    mean_field=False
)

In [ ]:
to_numpy = lambda x: x.detach().cpu().numpy()

t_list = [to_numpy(t) for t in data_target_kernel_many_batchs.array[:4, 0]]  # 4 maps  
u_list = [to_numpy(u) for u in u_kernel_many_batchs[:4, 0]]  # 4 maps

# Flatten target arrays to compute global vmin/vmax
mean_t = np.nanmean(t_list)
std_t = np.nanstd(t_list)

vmin = mean_t - 3 * std_t
vmax = mean_t + 3 * std_t

# --- Plot Target Maps ---
fig_target, axes = plt.subplots(2, 2, figsize=(8, 6))

for ax, t, i in zip(axes.flatten(), t_list, range(4)):
    im0 = ax.imshow(t, cmap="plasma", vmin=vmin, vmax=vmax)
    ax.axis("off")
    ax.set_title(f"Target {i+1}")

fig_target.colorbar(im0, ax=axes, shrink=0.7)
plt.suptitle("Target Maps")

# --- Plot Synthesis Maps ---
fig_synth, axes = plt.subplots(2, 2, figsize=(8, 6))

for ax, u, i in zip(axes.flatten(), u_list, range(4)):
    im0 = ax.imshow(u, cmap="plasma", vmin=vmin, vmax=vmax)
    ax.axis("off")
    ax.set_title(f"Synthesis {i+1}")

fig_synth.colorbar(im0, ax=axes, shrink=0.7)
plt.suptitle("Synthesis Maps")

plt.show()

Example code for performing synthesis from statistics

In [ ]:
im_target = im[:4, 0:1, :, :] # Target image with 4 maps and 1 channel

# Instantiate kernel data class on the target image
data_target_kernel_many_batchs = STL_2D_Kernel_Torch(im_target, pbc=True)
st_op_kernel_many_batchs = data_target_kernel_many_batchs.get_ST_op()

# Standardize target data
data_target_kernel_many_batchs_std, mean_target, std_target = st_op_kernel_many_batchs.wavelet_op.standardize(data_target_kernel_many_batchs, mean_field=False, inplace=False)

# Compute target statistics
target_stats_kernel_many_batchs = st_op_kernel_many_batchs.apply(data_target_kernel_many_batchs_std)

# Call user-friendly synthesis wrapper
u_kernel_many_batchs = synthesize_from_stats(
    target_stats_kernel_many_batchs,
    pbc_running=True,
    mean_target=mean_target,
    std_target=std_target,
    max_iter=100
)

#### PBC → Non PBC

In this section, we attempt to synthesize a **non-periodic image** from a **periodic target**. Two remarks:

- One dyadic scale is removed from the underlying ST operators of the synthesis to achieve better results.  
- The **crop method** used for estimating statistics with a non-PBC map is the default (`fully_flexible`). To use another crop method, the user-friendly wrapper does not allow it; you need to specify the crop method at a **mid-level synthesis function**.

In [ ]:
im_target = im[0, 0, :, :]

# Instantiate kernel data class on the target image
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)

# Call user-friendly synthesis wrapper
u_kernel_no_pbc = synthesize_from_maps(
    data_target_kernel,
    pbc_running=False,
    nbatch=1,
    max_iter=100,
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel.array, u_kernel_no_pbc, suptitle = "Kernel - PBC -> No PBC", titles=["Target - PBC", "Synthesis - No PBC"])

#### Non PBC → PBC

In [ ]:
im_target = im[0, 0, :128, :128] # No PBC target image

# Instantiate kernel data class on the target image
data_target_kernel_no_pbc = STL_2D_Kernel_Torch(im_target, pbc=False)

# Call user-friendly synthesis wrapper
u_kernel_pbc = synthesize_from_maps(
    data_target_kernel_no_pbc,
    pbc_running=True,
    nbatch=1
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel_no_pbc.array, u_kernel_pbc, suptitle = "Kernel - No PBC -> PBC", titles=["Target - No PBC", "Synthesized - PBC"])

#### Non PBC → Non PBC

In [ ]:
im_target = im[0, 0, :128, :128] # No PBC target image

# Instantiate kernel data class on the target image
data_target_kernel_no_pbc = STL_2D_Kernel_Torch(im_target, pbc=False)

# Call user-friendly synthesis wrapper
u_kernel_no_pbc = synthesize_from_maps(
    data_target_kernel_no_pbc,
    pbc_running=False,
    nbatch=1
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel_no_pbc.array, u_kernel_no_pbc, suptitle = "Kernel - No PBC -> No PBC", titles=["Target - No PBC", "Synthesis - No PBC"])

#### NaNs → NaNs

For syntheses whose running maps include a mask, it is preferable to increase the number of optimization iterations, as pixels near NaNs generally take longer to converge.

**Notes:** Optimization parameters can be overridden by the user in the `synthesize_from_maps` wrapper, for example the `max_iter` parameter for syntheses whose running maps include a mask.

In [ ]:
seed = 26
np.random.seed(seed)

im_target_nan = im[0, 0, :, :].copy()

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block 
im_target_nan[(np.random.rand(30)*im_target_nan.shape[0]).astype('int'),
    (np.random.rand(30)*im_target_nan.shape[0]).astype('int')] = np.nan
im_target_nan[50:80,50:80] = np.nan

# Instantiate kernel data class on NaN target image
data_target_kernel_nan = STL_2D_Kernel_Torch(array=im_target_nan, pbc=True)

# Call user-friendly synthesis wrapper
max_iter = 70 # overwrite default max_iter
u_kernel_nan = synthesize_from_maps(
    data_target_kernel_nan,
    nbatch=1,
    pbc_running=True,
    max_iter=max_iter
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel_nan.array, u_kernel_nan, suptitle = "Kernel - NaN to NaN (same mask)", titles=["Target - NaN", "Synthesis - NaN"])

Example code for performing synthesis from statistics

In [ ]:
seed = 26
np.random.seed(seed)

im_target_nan = im[0, 0, :, :].copy()

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block 
im_target_nan[(np.random.rand(30)*im_target_nan.shape[0]).astype('int'),
    (np.random.rand(30)*im_target_nan.shape[0]).astype('int')] = np.nan
im_target_nan[50:80,50:80] = np.nan

# Instantiate kernel data class on NaN target image
data_target_kernel_nan = STL_2D_Kernel_Torch(im_target_nan, pbc=True)
st_op_kernel_nan = data_target_kernel_nan.get_ST_op()

# Standardize target data
data_target_kernel_nan_std, mean_target, std_target = st_op_kernel_nan.wavelet_op.standardize(data_target_kernel_nan, mean_field=False, inplace=False)

# Compute target statistics
target_stats_kernel_nan = st_op_kernel_nan.apply(data_target_kernel_nan_std, compute_PS=False)

# Call user-friendly synthesis wrapper
u_kernel_nan = synthesize_from_stats(
    target_stats_kernel_nan,
    pbc_running=True,
    mean_target=mean_target,
    std_target=std_target,
    max_iter=70
)

# Plot target vs synthesis for NaN target and NaN synthesis (same mask)
plot_target_vs_synthesis(data_target_kernel_nan.array, u_kernel_nan, suptitle = "Kernel - NaN to NaN (same mask)", titles=["Target - NaN", "Synthesis - NaN"])

#### NaNs → NaNs (≠ Mask)

In [ ]:
seed = 26
np.random.seed(seed)

im_target_nan = im[0, 0, :, :].copy()

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block 
im_target_nan[(np.random.rand(30)*im_target_nan.shape[0]).astype('int'),
    (np.random.rand(30)*im_target_nan.shape[0]).astype('int')] = np.nan
im_target_nan[50:80,50:80] = np.nan

# Specifiy an array running mask different from the NaN mask of the target image
running_mask = np.zeros_like(im[0, 0, :, :])
running_mask[(np.random.rand(30)*running_mask.shape[0]).astype('int'),
    (np.random.rand(30)*running_mask.shape[0]).astype('int')] = np.nan
running_mask[150:175,150:175] = np.nan

# Instantiate kernel data class
data_target_kernel_nan = STL_2D_Kernel_Torch(im_target_nan, pbc=True)

# Call user-friendly synthesis wrapper
max_iter = 70 # overwrite default max_iter
u_kernel_nan_new_mask = synthesize_from_maps(
    data_target_kernel_nan,
    pbc_running=True,
    nbatch=1,
    running_mask=running_mask,
    max_iter=max_iter
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel_nan.array, u_kernel_nan_new_mask, suptitle = "Kernel - NaN to NaN (different mask)", titles=["Target - NaN", "Synthesized - NaN (new mask)"])

Example code for performing synthesis from statistics

In [ ]:
seed = 26
np.random.seed(seed)

im_target_nan = im[0, 0, :, :].copy()

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block 
im_target_nan[(np.random.rand(30)*im_target_nan.shape[0]).astype('int'),
    (np.random.rand(30)*im_target_nan.shape[0]).astype('int')] = np.nan
im_target_nan[50:80,50:80] = np.nan

# Specifiy an array running mask different from the NaN mask of the target image
running_mask = np.zeros_like(im[0, 0, :, :])
running_mask[(np.random.rand(30)*running_mask.shape[0]).astype('int'),
    (np.random.rand(30)*running_mask.shape[0]).astype('int')] = np.nan
running_mask[150:175,150:175] = np.nan

# Instantiate kernel data class on NaN target image
data_target_kernel_nan = STL_2D_Kernel_Torch(im_target_nan, pbc=True)
st_op_kernel_nan = data_target_kernel_nan.get_ST_op()

# Standardize target data
data_target_kernel_nan_std, mean_target, std_target = st_op_kernel_nan.wavelet_op.standardize(data_target_kernel_nan, mean_field=False, inplace=False)

# Compute target statistics
target_stats_kernel_nan = st_op_kernel_nan.apply(data_target_kernel_nan_std, compute_PS=False)

# Call user-friendly synthesis wrapper
u_kernel_nan_new_mask = synthesize_from_stats(
    target_stats_kernel_nan,
    pbc_running=True,
    running_mask=running_mask,
    mean_target=mean_target,
    std_target=std_target,
    max_iter=70
)

# Plot target vs synthesis for NaN target and NaN synthesis (same mask)
plot_target_vs_synthesis(data_target_kernel_nan.array, u_kernel_nan_new_mask, suptitle = "Kernel - NaN to NaN (same mask)", titles=["Target - NaN", "Synthesis - NaN"])

#### Non NaNs → NaNs

In [ ]:
im_target = im[0, 0, :, :]

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block to a NaN mask 
seed = 26
np.random.seed(seed)

# Instantiate kernel data class
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)

# Specifiy an array running mask
running_mask = np.zeros_like(im[0, 0, :, :])
running_mask[(np.random.rand(30)*running_mask.shape[0]).astype('int'),
    (np.random.rand(30)*running_mask.shape[0]).astype('int')] = np.nan
running_mask[50:80,50:80] = np.nan

# Call user-friendly synthesis wrapper
max_iter = 70 # overwrite default max_iter
u_kernel_nan = synthesize_from_maps(
    data_target_kernel,
    pbc_running=True,
    nbatch=1,
    running_mask=running_mask,
    max_iter=max_iter
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel.array, u_kernel_nan, suptitle = "Kernel - no NaN to NaN", titles=["Target", "Synthesized - NaN"])

Example code for performing synthesis from statistics

In [ ]:
im_target = im[0, 0, :, :]

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block to a NaN mask 
seed = 26
np.random.seed(seed)

# Instantiate kernel data class
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)

# Specifiy an array running mask
running_mask = np.zeros_like(im[0, 0, :, :])
running_mask[(np.random.rand(30)*running_mask.shape[0]).astype('int'),
    (np.random.rand(30)*running_mask.shape[0]).astype('int')] = np.nan
running_mask[50:80,50:80] = np.nan

# Instantiate kernel data class on NaN target image
data_target_kernel = STL_2D_Kernel_Torch(im_target, pbc=True)
st_op_kernel = data_target_kernel.get_ST_op()

# Standardize target data
data_target_kernel_std, mean_target, std_target = st_op_kernel.wavelet_op.standardize(data_target_kernel, mean_field=False, inplace=False)

# Compute target statistics
target_stats_kernel = st_op_kernel.apply(data_target_kernel_std, compute_PS=False)

# Call user-friendly synthesis wrapper
u_kernel_nan = synthesize_from_stats(
    target_stats_kernel,
    pbc_running=True,
    running_mask=running_mask,
    mean_target=mean_target,
    std_target=std_target,
    max_iter=70
)

# Plot target vs synthesis for NaN target and NaN synthesis (same mask)
plot_target_vs_synthesis(data_target_kernel.array, u_kernel_nan, suptitle = "Kernel - no NaN to NaN", titles=["Target - no NaN", "Synthesis - NaN"])

#### NaNs → Non NaNs

In [ ]:
seed = 26
np.random.seed(seed)

im_target_nan = im[0, 0, :, :].copy()

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block 
im_target_nan[(np.random.rand(30)*im_target_nan.shape[0]).astype('int'),
    (np.random.rand(30)*im_target_nan.shape[0]).astype('int')] = np.nan
im_target_nan[50:80,50:80] = np.nan

# Instantiate kernel data class
data_target_kernel_nan = STL_2D_Kernel_Torch(im_target_nan, pbc=True)

# Specifiy an empty running mask 
running_mask = np.zeros_like(im[0, 0, :, :])

# Call user-friendly synthesis wrapper
u_kernel_no_nan = synthesize_from_maps(
    data_target_kernel_nan,
    pbc_running=True,
    nbatch=1,
    running_mask=running_mask
)

In [ ]:
plot_target_vs_synthesis(data_target_kernel_nan.array, u_kernel_no_nan, suptitle = "Kernel - NaN to no NaN", titles=["Target - NaN", "Synthesized - no NaN"])

Example code for performing synthesis from statistics

In [ ]:
seed = 26
np.random.seed(seed)

im_target_nan = im[0, 0, :, :].copy()

# Add ~30 scattered NaN pixels and a 30x30 NaN pixel block 
im_target_nan[(np.random.rand(30)*im_target_nan.shape[0]).astype('int'),
    (np.random.rand(30)*im_target_nan.shape[0]).astype('int')] = np.nan
im_target_nan[50:80,50:80] = np.nan

# Instantiate kernel data class on NaN target image
data_target_kernel_nan = STL_2D_Kernel_Torch(im_target_nan, pbc=True)
st_op_kernel_nan = data_target_kernel_nan.get_ST_op()

# Standardize target data
data_target_kernel_nan_std, mean_target, std_target = st_op_kernel_nan.wavelet_op.standardize(data_target_kernel_nan, mean_field=False, inplace=False)

# Compute target statistics
target_stats_kernel_nan = st_op_kernel_nan.apply(data_target_kernel_nan_std, compute_PS=False)

# Call user-friendly synthesis wrapper
u_kernel = synthesize_from_stats(
    target_stats_kernel_nan,
    pbc_running=True,
    running_mask=np.zeros_like(im[0, 0, :, :]),
    mean_target=mean_target,
    std_target=std_target,
    max_iter=70
)

# Plot target vs synthesis for NaN target and NaN synthesis (same mask)
plot_target_vs_synthesis(data_target_kernel_nan.array, u_kernel, suptitle = "Kernel - NaN to no NaN", titles=["Target - NaN", "Synthesis - no NaN"])